# test_revised triple 추출 — 최종 모델(DREAM revised)

최종 채택 모델 `atlop_dream_revised.pt`로 `test_revised.json`의 관계 triple을 추출한다.
(test_revised에 gold labels가 있어 F1도 함께 계산)

**실행 전**: `Scripts/atlop/extract_triples.py`·`re_model_dream.py`·`docred_io.py`가 **dh에 push**돼 있어야 하고,
**학습된 `atlop_dream_revised.pt`가 Drive에** 있어야 함.


## 1) Google Drive 마운트

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2) 코드(dh) + revised 데이터(main, LFS) 가져오기

In [2]:
%cd /content
!apt-get -qq install -y git-lfs > /dev/null 2>&1
!git lfs install --skip-repo
!rm -rf project1
!git clone -q https://github.com/multiful/Information_Extraction.git project1
%cd /content/project1
!git checkout -q dh
# test_revised(추출 대상) + train_revised(Ign F1 필터용) 실체화
!git checkout origin/main -- docred_data/data/test_revised.json docred_data/data/train_revised.json
!git lfs pull --include="docred_data/data/test_revised.json,docred_data/data/train_revised.json"
!pip install -q transformers==4.57.6 accelerate

import re, pathlib
_p = pathlib.Path("data/docred_io.py"); _s = _p.read_text(encoding="utf-8")
if "test_revised" not in _s:
    _p.write_text(re.sub(r'SPLITS\s*=\s*\[[^\]]*\]',
        'SPLITS = ["train_annotated","train_distant","dev","test","train_revised","dev_revised","test_revised"]',
        _s, count=1), encoding="utf-8")
    print("docred_io SPLITS 런타임 패치")

import os
assert os.path.exists("Scripts/atlop/extract_triples.py"), "extract_triples.py 없음 -> dh push 확인!"
for f in ["test_revised", "train_revised"]:
    kb = os.path.getsize(f"docred_data/data/{f}.json") // 1024
    print(f"{f}: {kb} KB", "OK" if kb > 100 else "  <-- LFS 실체화 실패")

/content
Git LFS initialized.
/content/project1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 145.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
test_revised: 3132 KB OK
train_revised: 18166 KB OK


## 3) 인코더(bert-base-cased) 로컬 다운로드

In [3]:
import os
os.makedirs("/content/bert-base-cased", exist_ok=True)
%cd /content/bert-base-cased
B  = "https://huggingface.co/google-bert/bert-base-cased/resolve/main"
UA = "Mozilla/5.0"
for f in ["config.json", "vocab.txt", "tokenizer.json", "tokenizer_config.json", "model.safetensors"]:
    print("↓", f, flush=True)
    !wget -c --tries=200 --timeout=30 --waitretry=3 --header="User-Agent: {UA}" -O {f} {B}/{f}
%cd /content/project1
!ls -lh /content/bert-base-cased/model.safetensors

/content/bert-base-cased
↓ config.json
--2026-07-15 05:27:40--  https://huggingface.co/google-bert/bert-base-cased/resolve/main/config.json
Resolving huggingface.co (huggingface.co)... 18.238.49.112, 18.238.49.70, 18.238.49.10, ...
Connecting to huggingface.co (huggingface.co)|18.238.49.112|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/models/google-bert/bert-base-cased/cd5ef92a9fb2f889e972770a36d4ed042daf221e/config.json?%2Fgoogle-bert%2Fbert-base-cased%2Fresolve%2Fmain%2Fconfig.json=&etag=%22107460496b431545e4f921afb3fd5486fd2ae79d%22 [following]
--2026-07-15 05:27:40--  https://huggingface.co/api/resolve-cache/models/google-bert/bert-base-cased/cd5ef92a9fb2f889e972770a36d4ed042daf221e/config.json?%2Fgoogle-bert%2Fbert-base-cased%2Fresolve%2Fmain%2Fconfig.json=&etag=%22107460496b431545e4f921afb3fd5486fd2ae79d%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 570

## 4) triple 추출 (+ test F1)
- `--ckpt` : Drive의 학습된 체크포인트 경로 (본인 경로로!)
- `--split test_revised` / `--eval` : gold labels로 F1도 계산 (Ign 필터 = train_revised)
- 결과: `results/atlop_dream_revised_test_revised_triples_v2.json` — **리치 메타데이터** triple
  (head/relation/tail = {id,name,type/code}, confidence, source.is_revised, evidence 문장)

In [ ]:
CKPT = "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/atlop_dream_revised.pt"   # <- 본인 경로

# 주의: 리치 메타데이터 출력은 수정된 extract_triples.py가 dh에 push돼 있어야 함(2번 셀이 clone함)
!python -u -m Scripts.atlop.extract_triples \
  --ckpt "{CKPT}" \
  --model_name_or_path /content/bert-base-cased \
  --split test_revised --eval --eval_batch_size 32 \
  --out results/atlop_dream_revised_test_revised_triples_v2.json

## 5) 결과 미리보기 + Drive 백업

In [ ]:
import os, json, shutil
src = "results/atlop_dream_revised_test_revised_triples_v2.json"
tp = json.load(open(src, encoding="utf-8"))
print("총 triple:", len(tp),
      "| 문서 수:", len({t["source"]["document_id"] for t in tp}),
      "| 인적정제일치:", sum(1 for t in tp if t["source"]["is_revised"]))
for t in tp[:10]:
    h, r, ta = t["head"], t["relation"], t["tail"]
    print(f'  ({h["name"]}:{h["type"]}) -[{r["name"]} / {r["code"]}]-> ({ta["name"]}:{ta["type"]}) '
          f'conf={t["confidence"]} rev={t["source"]["is_revised"]}')

# Drive 저장 (shutil = 실패 시 에러로 바로 확인, !cp의 조용한 실패 방지)
dst_dir = "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"   # <- 본인 경로
assert os.path.exists(src), f"소스 없음: {src} (4번 추출 셀이 됐는지 확인)"
os.makedirs(dst_dir, exist_ok=True)
dst = os.path.join(dst_dir, os.path.basename(src))
shutil.copy(src, dst)
print("✅ Drive 저장됨:", os.path.exists(dst), "|", os.path.getsize(dst), "bytes")
print("   위치:", dst)